# 02 — Modélisation

Ce notebook compare trois modèles de régression pour prédire le prix des véhicules :

- DummyRegressor
- LinearRegression
- RandomForestRegressor

Les modèles seront entraînés sur les mêmes données et évalués avec les mêmes métriques.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/interim/vehicles_clean.csv")

df.head()

,YEAR,KM,BODYDOORS,PASSENGERS,MAKE,MODEL,BODYDESCF,DRIVETRAIN,FUEL,TRANSDESCF,COLORF,PRICE
0,2024,14243,4,5,Nissan,Versa,Berline,Traction avant,Essence,Automatique,Gris,18995
1,2021,58800,4,5,Ford,Escape,VUS,4 roues motrices,Essence,Automatique,Noir,18995
2,2019,85525,2,0,Chevrolet,Express Commercial Cutaway,Berline,Propulsion,Essence,Automatique,Blanc,26995
3,2023,60820,4,5,Hyundai,Venue,VUS,Traction avant,Essence,Automatique,Bleu,17995
4,2020,103329,4,5,Volkswagen,Jetta,Berline,Traction avant,Essence,Automatique,Argent,17495


In [3]:
df.shape

(543, 12)

## Séparation des variables explicatives et de la cible

In [4]:
X = df.drop(columns="PRICE")
y = df["PRICE"]

In [5]:
X.shape, y.shape

((543, 11), (543,))

## Séparation des données d'entraînement et de test

Les données sont séparées en deux groupes :

- 80 % pour entraîner les modèles ;
- 20 % pour évaluer leurs performances sur des données non utilisées pendant l'entraînement.

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [8]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (434, 11)
X_test  : (109, 11)
y_train : (434,)
y_test  : (109,)


## Préparation des variables

Les variables numériques sont conservées telles quelles.

Les variables catégorielles sont transformées avec `OneHotEncoder` afin de convertir les catégories en valeurs numériques utilisables par les modèles.

In [9]:
variables_numeriques = [
    "YEAR",
    "KM",
    "BODYDOORS",
    "PASSENGERS"
]

variables_categorielles = [
    "MAKE",
    "MODEL",
    "BODYDESCF",
    "DRIVETRAIN",
    "FUEL",
    "TRANSDESCF",
    "COLORF"
]

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [11]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            "passthrough",
            variables_numeriques
        )
    ]
)

In [12]:
X_train_prepared = preprocessor.fit_transform(X_train)

X_test_prepared = preprocessor.transform(X_test)

In [13]:
print("Avant préparation :", X_train.shape)
print("Après préparation :", X_train_prepared.shape)

print("Test avant :", X_test.shape)
print("Test après :", X_test_prepared.shape)

Avant préparation : (434, 11)
Après préparation : (434, 191)
Test avant : (109, 11)
Test après : (109, 191)


## Modèle 1 — DummyRegressor

Le `DummyRegressor` sert de modèle de référence.

Il effectue une prédiction très simple sans réellement apprendre les relations entre les variables.

Il permet de vérifier si les modèles plus avancés font réellement mieux qu'une stratégie naïve.

In [14]:
from sklearn.dummy import DummyRegressor

In [15]:
dummy_model = DummyRegressor(strategy="mean")

In [16]:
dummy_model.fit(X_train_prepared, y_train)

,strategy,'mean'
,constant,None
,quantile,None


In [17]:
y_pred_dummy = dummy_model.predict(X_test_prepared)

In [18]:
print("Prix moyen appris :", y_train.mean())

print("Premières prédictions :")
print(y_pred_dummy[:5])

Prix moyen appris : 23760.677419354837
Premières prédictions :
[23760.67741935 23760.67741935 23760.67741935 23760.67741935
 23760.67741935]


## Évaluation du DummyRegressor

Les performances sont évaluées avec trois métriques :

- **MAE** : erreur moyenne entre le prix réel et le prix prédit.
- **RMSE** : similaire à la MAE, mais pénalise davantage les grosses erreurs.
- **R²** : mesure la capacité du modèle à expliquer les variations du prix.

In [21]:
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

In [22]:
mae_dummy = mean_absolute_error(y_test, y_pred_dummy)
rmse_dummy = root_mean_squared_error(y_test, y_pred_dummy)
r2_dummy = r2_score(y_test, y_pred_dummy)

print("MAE  :", mae_dummy)
print("RMSE :", rmse_dummy)
print("R²   :", r2_dummy)

MAE  : 5631.577389760284
RMSE : 6864.37409126827
R²   : -0.040239363738684863


### Interprétation

Le DummyRegressor obtient une MAE d'environ 5 632 $ et un R² proche de 0.

Ce modèle sert de référence minimale. Les modèles suivants devront obtenir une erreur plus faible et un R² plus élevé pour être considérés comme plus performants.

## Modèle 2 — LinearRegression

La régression linéaire cherche une relation globale entre les caractéristiques du véhicule et son prix.

Elle sert de premier véritable modèle prédictif.

In [23]:
from sklearn.linear_model import LinearRegression

In [24]:
linear_model = LinearRegression()

linear_model.fit(X_train_prepared, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [25]:
y_pred_linear = linear_model.predict(X_test_prepared)

In [26]:
mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = root_mean_squared_error(y_test, y_pred_linear)
r2_linear = r2_score(y_test, y_pred_linear)

print("MAE  :", mae_linear)
print("RMSE :", rmse_linear)
print("R²   :", r2_linear)

MAE  : 2281.5564111370054
RMSE : 3336.207611950856
R²   : 0.754281681416473


### Interprétation

La régression linéaire améliore nettement les résultats par rapport au DummyRegressor.

- MAE : environ 2 282 $
- RMSE : environ 3 336 $
- R² : environ 0,75

Le modèle parvient donc à expliquer une grande partie des variations de prix à partir des caractéristiques des véhicules.

## Modèle 3 — RandomForestRegressor

Le RandomForestRegressor combine plusieurs arbres de décision.

Contrairement à la régression linéaire, il peut apprendre des relations plus complexes entre les caractéristiques du véhicule et son prix.

In [29]:
from sklearn.ensemble import RandomForestRegressor

On construit 100 arbres et on combine leurs prédictions.

In [30]:
random_forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

In [31]:
random_forest_model.fit(X_train_prepared, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [32]:
y_pred_forest = random_forest_model.predict(X_test_prepared)

In [33]:
mae_forest = mean_absolute_error(y_test, y_pred_forest)
rmse_forest = root_mean_squared_error(y_test, y_pred_forest)
r2_forest = r2_score(y_test, y_pred_forest)

print("MAE  :", mae_forest)
print("RMSE :", rmse_forest)
print("R²   :", r2_forest)

MAE  : 2563.302752293578
RMSE : 3574.4174478847767
R²   : 0.7179397154206767


### Interprétation du RandomForestRegressor

Le RandomForestRegressor améliore nettement les résultats par rapport au DummyRegressor.

Cependant, avec les paramètres actuels, ses performances sont légèrement inférieures à celles de la régression linéaire.

Cela montre qu'un modèle plus complexe n'est pas nécessairement plus performant pour un jeu de données donné.

## Comparaison des modèles

In [34]:
resultats = pd.DataFrame({
    "Modèle": [
        "DummyRegressor",
        "LinearRegression",
        "RandomForestRegressor"
    ],
    "MAE": [
        mae_dummy,
        mae_linear,
        mae_forest
    ],
    "RMSE": [
        rmse_dummy,
        rmse_linear,
        rmse_forest
    ],
    "R²": [
        r2_dummy,
        r2_linear,
        r2_forest
    ]
})

resultats

,Modèle,MAE,RMSE,R²
0,DummyRegressor,5631.577390,6864.374091,-0.040239
1,LinearRegression,2281.556411,3336.207612,0.754282
2,RandomForestRegressor,2563.302752,3574.417448,0.717940


## Validation croisée

Une validation croisée à 5 plis est utilisée pour vérifier si les performances des modèles restent stables sur différentes séparations des données.

In [35]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

In [36]:
pipeline_linear = Pipeline([
    ("preparation", preprocessor),
    ("modele", LinearRegression())
])

In [37]:
scores_linear = cross_val_score(
    pipeline_linear,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_linear)
print("R² moyen   :", scores_linear.mean())

R² par pli : [0.47061488 0.81227812 0.7383836  0.75931461 0.83909395]
R² moyen   : 0.7239370321268709


### Interprétation de la validation croisée

La régression linéaire obtient un R² moyen d'environ 0,72 sur les 5 plis.

Les résultats varient selon les groupes de test, mais restent globalement satisfaisants. Cela confirme que les performances observées avec la séparation train/test initiale ne dépendent pas uniquement d'une séparation favorable.

In [38]:
pipeline_forest = Pipeline([
    ("preparation", preprocessor),
    ("modele", RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [39]:
scores_forest = cross_val_score(
    pipeline_forest,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_forest)
print("R² moyen   :", scores_forest.mean())

R² par pli : [0.35464929 0.72559289 0.75754568 0.75505875 0.78250133]
R² moyen   : 0.6750695897070191


### Interprétation de la validation croisée du RandomForestRegressor

Le RandomForestRegressor obtient un R² moyen d'environ 0,68 sur les 5 plis.

Les performances sont globalement bonnes, mais légèrement inférieures à celles de la régression linéaire, qui obtient un R² moyen d'environ 0,72.

Cela confirme que, pour ce jeu de données et avec les paramètres actuels, la régression linéaire généralise mieux que le RandomForestRegressor.

# Expérimentation des variantes

## Variante du DummyRegressor

Une deuxième stratégie naïve est testée en utilisant la médiane des prix au lieu de la moyenne.

In [40]:
dummy_median = DummyRegressor(strategy="median")

dummy_median.fit(X_train_prepared, y_train)

y_pred_dummy_median = dummy_median.predict(X_test_prepared)

In [41]:
mae_dummy_median = mean_absolute_error(y_test, y_pred_dummy_median)
rmse_dummy_median = root_mean_squared_error(y_test, y_pred_dummy_median)
r2_dummy_median = r2_score(y_test, y_pred_dummy_median)

print("MAE  :", mae_dummy_median)
print("RMSE :", rmse_dummy_median)
print("R²   :", r2_dummy_median)

MAE  : 5474.311926605505
RMSE : 6755.622870392668
R²   : -0.007539755552781857


### Comparaison des stratégies Dummy

La stratégie utilisant la médiane obtient des résultats légèrement meilleurs que celle utilisant la moyenne.

Cependant, les deux modèles restent des références naïves avec un R² proche de 0.

## Ridge

Ridge est une régression linéaire avec régularisation.

La régularisation réduit l'importance excessive de certains coefficients afin de limiter le surapprentissage.

Deux valeurs de `alpha` seront testées :

- `alpha = 1` : régularisation modérée ;
- `alpha = 10` : régularisation plus forte.

In [42]:
from sklearn.preprocessing import StandardScaler

preprocessor_scaled = ColumnTransformer(
    transformers=[
        (
            "categories",
            OneHotEncoder(handle_unknown="ignore"),
            variables_categorielles
        ),
        (
            "numeriques",
            StandardScaler(),
            variables_numeriques
        )
    ]
)

In [43]:
from sklearn.linear_model import Ridge

In [44]:
ridge_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=1))
])

ridge_1.fit(X_train, y_train)

y_pred_ridge_1 = ridge_1.predict(X_test)

In [45]:
mae_ridge_1 = mean_absolute_error(y_test, y_pred_ridge_1)
rmse_ridge_1 = root_mean_squared_error(y_test, y_pred_ridge_1)
r2_ridge_1 = r2_score(y_test, y_pred_ridge_1)

print("MAE  :", mae_ridge_1)
print("RMSE :", rmse_ridge_1)
print("R²   :", r2_ridge_1)

MAE  : 2040.037855702886
RMSE : 2897.9153589603898
R²   : 0.8146029624574989


In [46]:
ridge_10 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=10))
])

ridge_10.fit(X_train, y_train)

y_pred_ridge_10 = ridge_10.predict(X_test)

In [47]:
mae_ridge_10 = mean_absolute_error(y_test, y_pred_ridge_10)
rmse_ridge_10 = root_mean_squared_error(y_test, y_pred_ridge_10)
r2_ridge_10 = r2_score(y_test, y_pred_ridge_10)

print("MAE  :", mae_ridge_10)
print("RMSE :", rmse_ridge_10)
print("R²   :", r2_ridge_10)

MAE  : 2613.575591640975
RMSE : 3380.1512277097004
R²   : 0.7477659814616573


### Comparaison des variantes Ridge

Ridge avec `alpha=1` obtient de meilleures performances que la régression linéaire classique.

Avec `alpha=10`, les performances diminuent. Une régularisation trop forte peut donc limiter excessivement le modèle.

Dans cette expérimentation, `alpha=1` donne le meilleur résultat parmi les variantes Ridge testées.

## Lasso

Lasso est une régression linéaire avec régularisation.

Contrairement à Ridge, Lasso peut réduire certains coefficients jusqu'à zéro, ce qui peut éliminer l'influence de certaines variables.

Deux valeurs de `alpha` seront testées :

- `alpha = 1`
- `alpha = 10`

In [48]:
from sklearn.linear_model import Lasso

In [49]:
lasso_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=1,
        max_iter=10000
    ))
])

lasso_1.fit(X_train, y_train)

y_pred_lasso_1 = lasso_1.predict(X_test)

In [50]:
mae_lasso_1 = mean_absolute_error(y_test, y_pred_lasso_1)
rmse_lasso_1 = root_mean_squared_error(y_test, y_pred_lasso_1)
r2_lasso_1 = r2_score(y_test, y_pred_lasso_1)

print("MAE  :", mae_lasso_1)
print("RMSE :", rmse_lasso_1)
print("R²   :", r2_lasso_1)

MAE  : 1858.4480838040429
RMSE : 2888.57386190385
R²   : 0.8157962992387754


### Interprétation de Lasso alpha=1

Lasso avec `alpha=1` obtient de très bonnes performances.

Son erreur moyenne est plus faible que celle de la régression linéaire et de Ridge avec `alpha=1`.

À ce stade, il s'agit du meilleur résultat obtenu sur le jeu de test.

In [51]:
lasso_10 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=10,
        max_iter=10000
    ))
])

lasso_10.fit(X_train, y_train)

y_pred_lasso_10 = lasso_10.predict(X_test)

In [52]:
mae_lasso_10 = mean_absolute_error(y_test, y_pred_lasso_10)
rmse_lasso_10 = root_mean_squared_error(y_test, y_pred_lasso_10)
r2_lasso_10 = r2_score(y_test, y_pred_lasso_10)

print("MAE  :", mae_lasso_10)
print("RMSE :", rmse_lasso_10)
print("R²   :", r2_lasso_10)

MAE  : 2112.9528652154418
RMSE : 2896.0038535366634
R²   : 0.814847462740958


### Comparaison des variantes Lasso

Les deux variantes Lasso obtiennent de bonnes performances.

`alpha=1` présente toutefois une erreur moyenne plus faible que `alpha=10`.

Une régularisation plus forte n'améliore donc pas les performances dans cette expérimentation.

## Variantes du RandomForest

In [53]:
random_forest_200 = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

random_forest_200.fit(X_train_prepared, y_train)

y_pred_forest_200 = random_forest_200.predict(X_test_prepared)

In [54]:
mae_forest_200 = mean_absolute_error(y_test, y_pred_forest_200)
rmse_forest_200 = root_mean_squared_error(y_test, y_pred_forest_200)
r2_forest_200 = r2_score(y_test, y_pred_forest_200)

print("MAE  :", mae_forest_200)
print("RMSE :", rmse_forest_200)
print("R²   :", r2_forest_200)

MAE  : 2562.110275229358
RMSE : 3552.1827447778146
R²   : 0.721437919380693


### Effet du nombre d'arbres

Le passage de 100 à 200 arbres améliore très légèrement les performances du RandomForestRegressor.

La différence reste toutefois faible. Augmenter le nombre d'arbres n'apporte donc pas d'amélioration importante dans cette expérimentation.

In [55]:
random_forest_depth_10 = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42
)

random_forest_depth_10.fit(X_train_prepared, y_train)

y_pred_forest_depth_10 = random_forest_depth_10.predict(X_test_prepared)

In [56]:
mae_forest_depth_10 = mean_absolute_error(
    y_test,
    y_pred_forest_depth_10
)

rmse_forest_depth_10 = root_mean_squared_error(
    y_test,
    y_pred_forest_depth_10
)

r2_forest_depth_10 = r2_score(
    y_test,
    y_pred_forest_depth_10
)

print("MAE  :", mae_forest_depth_10)
print("RMSE :", rmse_forest_depth_10)
print("R²   :", r2_forest_depth_10)

MAE  : 2601.502537017324
RMSE : 3609.6539468246697
R²   : 0.7123512214396199


### Effet de la profondeur des arbres

Limiter la profondeur des arbres à 10 n'améliore pas les performances du RandomForestRegressor.

Les erreurs augmentent légèrement et le R² diminue. Dans cette expérimentation, la limitation de profondeur n'apporte donc pas de bénéfice.

## Validation croisée de Ridge

La validation croisée à 5 plis permet de vérifier si les bonnes performances de Ridge avec `alpha=1` restent stables sur plusieurs séparations des données.

In [57]:
pipeline_ridge_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Ridge(alpha=1))
])

In [58]:
scores_ridge_1 = cross_val_score(
    pipeline_ridge_1,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_ridge_1)
print("R² moyen   :", scores_ridge_1.mean())

R² par pli : [0.48068579 0.84057827 0.84999682 0.81267433 0.88511704]
R² moyen   : 0.7738104535194067


### Interprétation de la validation croisée de Ridge

Ridge avec `alpha=1` obtient un R² moyen d'environ 0,77 sur les 5 plis.

Ce résultat est supérieur à celui de la régression linéaire classique et confirme que la régularisation améliore les performances générales du modèle.

In [59]:
pipeline_lasso_1 = Pipeline([
    ("preparation", preprocessor_scaled),
    ("modele", Lasso(
        alpha=1,
        max_iter=10000
    ))
])

In [60]:
scores_lasso_1 = cross_val_score(
    pipeline_lasso_1,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("R² par pli :", scores_lasso_1)
print("R² moyen   :", scores_lasso_1.mean())

R² par pli : [0.41552863 0.82465771 0.89257972 0.84808454 0.83510103]
R² moyen   : 0.7631903255545313


### Interprétation de la validation croisée de Lasso

Lasso avec `alpha=1` obtient un R² moyen d'environ 0,76 sur les 5 plis.

Ce résultat confirme les bonnes performances du modèle. Cependant, Ridge avec `alpha=1` obtient un R² moyen légèrement supérieur, autour de 0,77.

Les deux modèles régularisés améliorent les performances par rapport à la régression linéaire classique.

## Comparaison finale des expériences

Les différentes variantes sont regroupées afin de comparer leurs performances sur le même jeu de test.

La validation croisée est présentée pour les principaux modèles retenus.

In [61]:
resultats_finaux = pd.DataFrame({
    "Modèle": [
        "Dummy - Mean",
        "Dummy - Median",
        "LinearRegression",
        "Ridge alpha=1",
        "Ridge alpha=10",
        "Lasso alpha=1",
        "Lasso alpha=10",
        "RandomForest - 100 arbres",
        "RandomForest - 200 arbres",
        "RandomForest - max_depth=10"
    ],

    "MAE": [
        mae_dummy,
        mae_dummy_median,
        mae_linear,
        mae_ridge_1,
        mae_ridge_10,
        mae_lasso_1,
        mae_lasso_10,
        mae_forest,
        mae_forest_200,
        mae_forest_depth_10
    ],

    "RMSE": [
        rmse_dummy,
        rmse_dummy_median,
        rmse_linear,
        rmse_ridge_1,
        rmse_ridge_10,
        rmse_lasso_1,
        rmse_lasso_10,
        rmse_forest,
        rmse_forest_200,
        rmse_forest_depth_10
    ],

    "R² test": [
        r2_dummy,
        r2_dummy_median,
        r2_linear,
        r2_ridge_1,
        r2_ridge_10,
        r2_lasso_1,
        r2_lasso_10,
        r2_forest,
        r2_forest_200,
        r2_forest_depth_10
    ],

    "R² CV moyen": [
        np.nan,
        np.nan,
        scores_linear.mean(),
        scores_ridge_1.mean(),
        np.nan,
        scores_lasso_1.mean(),
        np.nan,
        scores_forest.mean(),
        np.nan,
        np.nan
    ]
})

resultats_finaux.round(3)

,Modèle,MAE,RMSE,R² test,R² CV moyen
0,Dummy - Mean,5631.577,6864.374,-0.040,NaN
1,Dummy - Median,5474.312,6755.623,-0.008,NaN
2,LinearRegression,2281.556,3336.208,0.754,0.724
3,Ridge alpha=1,2040.038,2897.915,0.815,0.774
4,Ridge alpha=10,2613.576,3380.151,0.748,NaN
5,Lasso alpha=1,1858.448,2888.574,0.816,0.763
6,Lasso alpha=10,2112.953,2896.004,0.815,NaN
7,RandomForest - 100 arbres,2563.303,3574.417,0.718,0.675
8,RandomForest - 200 arbres,2562.110,3552.183,0.721,NaN
9,RandomForest - max_depth=10,2601.503,3609.654,0.712,NaN


# Conclusion

L'objectif de ce TP était de prédire le prix d'un véhicule à partir de plusieurs caractéristiques comme l'année, le kilométrage, la marque, le modèle, le type de carrosserie, le carburant et la transmission.

Un premier modèle de référence avec `DummyRegressor` a permis d'établir une performance minimale. Ses résultats montrent qu'une prédiction basée uniquement sur la moyenne ou la médiane des prix produit une erreur importante et un R² proche de 0.

La `LinearRegression` améliore fortement les résultats et montre qu'une partie importante du prix peut être expliquée par une relation linéaire entre les caractéristiques des véhicules et leur prix.

Les modèles régularisés `Ridge` et `Lasso` donnent les meilleurs résultats de l'expérimentation. `Lasso` avec `alpha=1` obtient la plus faible erreur sur le jeu de test, avec une MAE d'environ 1 858 $ et un R² d'environ 0,82.

La validation croisée apporte toutefois une information supplémentaire. `Ridge` avec `alpha=1` obtient un R² moyen d'environ 0,77, légèrement supérieur à celui de `Lasso` avec `alpha=1`, qui obtient environ 0,76. Cela montre qu'il est important de ne pas évaluer un modèle uniquement sur une seule séparation train/test.

Le `RandomForestRegressor` obtient également de bonnes performances, mais reste inférieur aux modèles Ridge et Lasso dans cette expérimentation. L'augmentation du nombre d'arbres de 100 à 200 apporte peu d'amélioration, et limiter la profondeur des arbres à 10 réduit légèrement les performances.

En conclusion, les expérimentations montrent que la régularisation améliore les performances de la régression linéaire sur ce jeu de données. `Lasso(alpha=1)` est particulièrement performant sur le jeu de test, tandis que `Ridge(alpha=1)` présente une meilleure performance moyenne en validation croisée. Ces résultats montrent également qu'un modèle plus complexe n'est pas automatiquement plus performant et que plusieurs méthodes d'évaluation sont nécessaires pour comparer correctement les modèles.